In [1]:
import numpy as np
import pandas as pd

from thefuzz import process, fuzz

In [ ]:
data = {
    'loan_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'city': [
        ' Dhaka',       # শুরুতে স্পেস
        'dhaka ',       # শেষে স্পেস
        'DHAKA',        # সব বড় হাতের অক্ষর
        'chittagong',   # সঠিক বানান
        'chitagonj',    # বানান ভুল
        'chatgram',     # ভিন্ন উচ্চারণ/বানান
        'sylhet',       # সঠিক বানান
        'sylhett'       # টাইপো
    ]
}
df = pd.DataFrame(data)

df

,loan_id,city
0,101,Dhaka
1,102,dhaka
2,103,DHAKA
3,104,chittagong
4,105,chitagonj
5,106,chatgram
6,107,sylhet
7,108,sylhett


In [3]:
# unique cities

cities = df['city'].unique()
np.sort(cities)

array([' Dhaka', 'DHAKA', 'chatgram', 'chitagonj', 'chittagong', 'dhaka ',
       'sylhet', 'sylhett'], dtype=object)

## Preliminary Text Pre-processing

In [4]:
# city column er shob string ke lower case e convert kora
df['city'] = df['city'].str.lower()

# city column er shob string er shuru o sesh theke space remove kora
df['city'] = df['city'].str.strip()


np.sort(df['city'].unique())

array(['chatgram', 'chitagonj', 'chittagong', 'dhaka', 'sylhet',
       'sylhett'], dtype=object)

In [5]:
df

,loan_id,city
0,101,dhaka
1,102,dhaka
2,103,dhaka
3,104,chittagong
4,105,chitagonj
5,106,chatgram
6,107,sylhet
7,108,sylhett


## Fuzzy Matching

In [ ]:
'''
process.extract() er syntax:

process.extract(
    query,     
    choices,    
    limit,      
    scorer   
)

query: ami jei word er sathe match kore dekhte chai je ei word er similar r ki ki word ase.
choices: dataset er jei column er data theke matching dekhte chai oita choices. ekhane kintu ekta column er shob data newa hoitese na.
লাখ লাখ ডেটার ওপর কখনোই সরাসরি Fuzzy Matching চালানো হয় না। তোমার কোডে df['city'].unique() ব্যবহার করা হয়েছে। ১০ লক্ষ ডেটার মধ্যে হয়তো ইউনিক শহরের নাম থাকবে মাত্র ৩০০টি। অ্যালগরিদম শুধু ওই ৩০০টি শব্দের ওপর কাজ করবে, ১০ লক্ষ লাইনের ওপর নয়।

limit: ami koto gula match chai oita limit.
Usage of limit: ইন্ডাস্ট্রিতে limit মূলত Exploratory Data Analysis (EDA) এর সময় ডেটার অবস্থা ম্যানুয়ালি চেক করার জন্য ব্যবহৃত হয়। প্রোডাকশন লেভেলে (যখন কোড অটোমেটিকভাবে সার্ভারে চলে), তখন limit ব্যবহার না করে একটি নির্দিষ্ট threshold (যেমন: score >= 80) সেট করে দেওয়া হয়, যাতে ওই স্কোরের ওপরে থাকা সব ইনকনসিস্টেন্ট ডেটা স্বয়ংক্রিয়ভাবে রিপ্লেস হয়ে যায়।

১. limit কি সবসময় ৩ ব্যবহার করবো?
না। limit এর মান সবসময় ৩ হবে এমন কোনো নিয়ম নেই।
limit মানে হলো "Top N Results"।
ধরো, তোমার ডেটাসেটে ৫,০০০ ইউনিক শহরের নাম আছে। তুমি যদি limit ব্যবহার না করো, তবে process.extract() ৫,০০০টি শহরের সাথেই টার্গেট শব্দের স্কোরের একটি বিশাল লিস্ট প্রিন্ট করবে, যা দেখে তুমি কিছুই বুঝতে পারবে না।
limit=3 দিলে এটি শুধু সবচেয়ে কাছাকাছি থাকা সেরা ৩টি (Top 3) রেজাল্ট দেখাবে। তোমার প্রয়োজন অনুযায়ী তুমি limit=5, limit=10 বা limit=20 ব্যবহার করতে পারো। এটি শুধু আউটপুট কতটুকু দেখাবে, তা কন্ট্রোল করে।

২. Threshold জিনিসটা কী?
Threshold হলো একটি কাট-অফ পয়েন্ট বা পাসিং মার্ক।
Fuzzy matching অ্যালগরিদম প্রতিটি শব্দের জন্য 0 থেকে 100 এর মধ্যে একটি স্কোর দেয়। তুমি যদি ঠিক করে দাও যে, "স্কোর ৮০ বা তার বেশি হলেই কেবল আমি শব্দটিকে সঠিক হিসেবে ধরে নেব এবং রিপ্লেস করবো"—তবে এই ৮০-ই হলো তোমার Threshold।

scorer: 
২. scorer=fuzz.token_sort_ratio কেন ব্যবহার করা হলো?Fuzzy matching মূলত Levenshtein Distance (দুটি শব্দের মধ্যে কতগুলো ক্যারেক্টার পরিবর্তন করতে হবে তার গাণিতিক দূরত্ব) ব্যবহার করে কাজ করে। thefuzz লাইব্রেরিতে বেশ কয়েকটি আলাদা scorer আছে, যাদের কাজের ধরন ভিন্ন:fuzz.ratio (Strict Match): এটি একদম হুবহু ক্যারেক্টার বাই ক্যারেক্টার অর্ডার মিলিয়ে দেখে।উদাহরণ: "south korea" এবং "korea south" এর স্কোর আসবে অনেক কম (প্রায় 50), কারণ শব্দের অর্ডার (ক্রম) আলাদা।fuzz.partial_ratio: এটি সাবস্ট্রিং বা আংশিক ম্যাচ খোঁজে।উদাহরণ: "chittagong" এবং "chittagong city" এর স্কোর অনেক হাই আসবে।fuzz.token_sort_ratio (Industry Standard for Categorical Text): এটি প্রথমে পুরো স্ট্রিংটিকে ছোট ছোট শব্দে (token) ভাঙে, এরপর সেগুলোকে বর্ণমালা অনুযায়ী (alphabetically) সাজায়, এবং তারপর তুলনা করে।উদাহরণ: "south korea" এবং "korea south" কে সে প্রথমে ভেঙে ['korea', 'south'] বানাবে। এরপর উভয়ের স্কোর আসবে 100।fuzz.token_set_ratio: এটি ডুপ্লিকেট শব্দগুলোকে বাদ দিয়ে তারপর তুলনা করে। (যেমন: "dhaka dhaka city" এবং "dhaka city" এর স্কোর 100 আসবে)।কেন এটি ব্যবহার করা হলো?
ক্যাটাগরিক্যাল ডেটা এন্ট্রির সময় মানুষ প্রায়ই শব্দের অর্ডার উল্টে ফেলে (যেমন: "New York" কে "York New" লেখা, বা "Dhaka South" কে "South Dhaka" লেখা)। এই ধরনের ডেটা এন্ট্রির ভুল ঠিক করার জন্য token_sort_ratio সবচেয়ে বেশি নির্ভরযোগ্য এবং এটিই ইন্ডাস্ট্রিতে ডেটা ক্লিনিংয়ের জন্য স্ট্যান্ডার্ড হিসেবে ব্যবহৃত হয়।  

উদাহরণ:
process.extract(
    "chittagong",
    df["city"].unique(),
    limit=3,
    scorer=fuzz.token_sort_ratio
)
'''

In [ ]:


# 'chittagong' এর কাছাকাছি টপ ৩টি ম্যাচ খুঁজে বের করা
# token_sort_ratio শব্দের ভেতরের ক্যারেক্টারগুলোর অর্ডার ইগনোর করে ম্যাচিং করে
matches = process.extract(
    "chittagong", 
    df['city'].unique(), 
    limit=3, 
    scorer=fuzz.token_sort_ratio
)

print("\nFuzzy Matches for 'chittagong':")
print(matches)
# Output: [('chittagong', 100), ('chitagonj', 74), ('chatgram', 67)]


Fuzzy Matches for 'chittagong':
[('chittagong', 100), ('chitagonj', 84), ('chatgram', 44)]


## dictionary mapping

In [8]:
# বেস্ট প্র্যাকটিস: ম্যানুয়াল বা অটোমেটেড ম্যাপিং ডিকশনারি তৈরি করা
city_mapping = {
    'chitagonj': 'chittagong',
    'chatgram': 'chittagong',
    'sylhett': 'sylhet'
}

# এক লাইনে এবং অত্যন্ত দ্রুতগতিতে পুরো ডেটাসেট রিপ্লেস করা
df['city'] = df['city'].replace(city_mapping)

In [9]:
df

,loan_id,city
0,101,dhaka
1,102,dhaka
2,103,dhaka
3,104,chittagong
4,105,chittagong
5,106,chittagong
6,107,sylhet
7,108,sylhet


# concept

In [7]:
'''
১. Identifying the Problem
==========================

Real-world dataset-এ মানুষ manually data entry করলে একই value-এর
বিভিন্ন variation তৈরি হতে পারে।

উদাহরণ:

    'Germany'
    ' Germany'
    'germany'

অথবা:

    'New Zealand'
    ' New Zealand'

এগুলো মানুষের কাছে একই country মনে হলেও, Computer-এর কাছে এগুলো
সম্পূর্ণ আলাদা string।

সমস্যাটি identify করার জন্য Pandas-এর unique() এবং sort() ব্যবহার করে
একটি column-এর সব unique value দেখা যায়:

    professors['Country'].unique()
    sorted(professors['Country'].unique())

এভাবে একই data-এর বিভিন্ন inconsistent form সহজে খুঁজে বের করা যায়।


২. কেন এটি Machine Learning-এর জন্য Problem?
=============================================

Machine Learning model string-এর meaning বুঝতে পারে না।

Model-এর কাছে:

    'Germany'
    ' germany'
    'germany'

তিনটি আলাদা value।

অর্থাৎ একই category বিভিন্ন নামে থাকলে model সেই data-কে আলাদা
category হিসেবে treat করবে।

ফলে একই category-এর data বিভিন্ন জায়গায় split হয়ে যায় এবং model
সঠিক pattern শিখতে পারে না।


৩. Preliminary Text Pre-processing — The 80% Fix
================================================

Advanced technique ব্যবহার করার আগে সাধারণ text inconsistency দূর করার
জন্য দুটি basic operation খুব কার্যকর:

    ১. lower()
       সব character-কে lowercase করে।

    ২. strip()
       string-এর শুরু এবং শেষে থাকা unnecessary whitespace remove করে।

উদাহরণ:

    ' Germany ' → 'germany'
    ' NEW ZEALAND ' → 'new zealand'

এগুলো ব্যবহার করলে অনেক common data-entry inconsistency সহজেই দূর করা যায়।

উদাহরণ:

    professors['Country'] = (
        professors['Country']
        .str.lower()
        .str.strip()
    )


৪. lower() এবং strip() করার পরও Problem থাকতে পারে
====================================================

Basic preprocessing করার পরেও কিছু inconsistency থেকে যেতে পারে।

উদাহরণ:

    'south korea'
    'southkorea'

এখানে মাঝখানের space-এর কারণে দুইটি আলাদা string তৈরি হয়েছে।

আবার spelling mistake থাকতে পারে:

    'ausrtalia'
    'australia'

মানুষ বুঝতে পারে এগুলো একই country বোঝাচ্ছে, কিন্তু Machine Learning
model এগুলোকে আলাদা category হিসেবে দেখবে।

Real-world dataset-এ হাজার বা লাখ লাখ record থাকতে পারে। তাই প্রতিটি
value manually check করে hardcode করা practical নয়।


৫. Fuzzy Matching কী?
=====================

Fuzzy Matching হলো এমন একটি technique, যার মাধ্যমে দেখতে বা লিখতে
প্রায় একই রকম text string automatically খুঁজে বের করা যায়।

এটি exact matching-এর মতো শুধু "একই কি না" দেখে না; বরং দুটি string-এর
মধ্যে কতটা similarity আছে সেটিও হিসাব করে।

উদাহরণ:

    'south korea'
    'southkorea'

দুটি string exactly same নয়, কিন্তু তাদের মধ্যে similarity অনেক বেশি।


৬. Edit Distance কী?
====================

Fuzzy Matching-এর একটি গুরুত্বপূর্ণ ধারণা হলো Edit Distance।

একটি string-কে অন্য string-এ convert করতে কতগুলো basic operation
প্রয়োজন হয়, সেটিই Edit Distance-এর ধারণা।

সাধারণ operation:

    - Add / Insert → character যোগ করা
    - Remove / Delete → character মুছে ফেলা
    - Replace / Substitute → একটি character পরিবর্তন করা

উদাহরণ:

    'apple' → 'snapple'

এখানে 's' এবং 'n' যোগ করতে হয়।

তাই এই transformation-এর জন্য ২টি insertion operation প্রয়োজন।


৭. Fuzzy Matching Score / Ratio
================================

Fuzzy Matching library সাধারণত দুটি string-এর similarity বোঝানোর জন্য
একটি score বা ratio দেয়।

অনেক library-তে এই score 0 থেকে 100-এর মধ্যে থাকে।

    Score = 100
    → দুটি string খুব closely match করে / exact match হতে পারে।

    Score কম
    → দুটি string-এর মধ্যে similarity কম।

অর্থাৎ score যত বেশি, string দুটির similarity সাধারণত তত বেশি।

উদাহরণ:

    'south korea'
    'southkorea'

এদের মধ্যে high similarity পাওয়া যেতে পারে, যদিও strings দুটি
exactly একই নয়।


৮. replace_matches_in_column() কী করে?
=======================================

Fuzzy Matching ব্যবহার করে inconsistent value automatically ঠিক করার
জন্য একটি custom function তৈরি করা যায়।

যেমন:

    replace_matches_in_column()

এই function-এর basic idea হলো:

    ১. একটি correct/reference value দেওয়া হবে।
       যেমন: 'south korea'

    ২. একটি minimum similarity threshold বা min_ratio দেওয়া হবে।
       যেমন: 47

    ৩. Dataset-এর অন্যান্য string-এর সাথে reference value-এর similarity
       score calculate করা হবে।

    ৪. যেসব string-এর score min_ratio-এর সমান বা তার বেশি হবে,
       সেগুলোকে reference value দিয়ে replace করা হবে।

অর্থাৎ:

    'south korea'       → reference value

    'southkorea'        → high similarity
                          ↓
                       'south korea'

এভাবে manually প্রতিটি ভুল value ঠিক না করেও অনেক inconsistent
value automatically standardize করা যায়।


৯. Fuzzy Matching-এর Limitation
================================

Fuzzy Matching powerful হলেও এর ওপর 100% blindly depend করা উচিত নয়। কারণ similarity বেশি হলেই যে দুটি value অবশ্যই একই category বোঝাবে,
তা নয়। বিশেষ করে threshold বা min_ratio খুব কম সেট করলে false match হতে পারে।

উদাহরণ:
    'austria'
    'australia'

দুটি country-এর spelling অনেকটা কাছাকাছি।

Threshold খুব low হলে algorithm ভুলভাবে ধরে নিতে পারে যে এগুলো একই value এবং একটি value-কে অন্যটি দিয়ে replace করে দিতে পারে।


১০. Threshold / min_ratio কী?
=============================

min_ratio হলো একটি minimum similarity threshold।

উদাহরণ:

    min_ratio = 80

এর অর্থ হলো শুধু যেসব string-এর similarity score 80 বা তার বেশি,
সেগুলোকে match হিসেবে consider করা হবে।

    High threshold
    → ভুল match হওয়ার সম্ভাবনা কম
    → কিন্তু কিছু valid similar value miss হতে পারে

    Low threshold
    → বেশি variation match হবে
    → কিন্তু false match হওয়ার ঝুঁকি বাড়বে

তাই threshold dataset-এর context অনুযায়ী নির্বাচন করতে হবে।


১১. Sanity Check কেন দরকার?
===========================

Fuzzy Matching automatically data পরিবর্তন করে বলে output verify করা
খুব গুরুত্বপূর্ণ। automation করার পরও একবার manually output verify করা
best practice। এটিকে Sanity Check বলা যায়।



১৩. Overall Data Preprocessing Workflow
=======================================

Categorical text-এর inconsistency handle করার একটি practical workflow:

    Step 1:
    Unique values identify করো
            ↓
    Step 2:
    lower() দিয়ে case standardize করো
            ↓
    Step 3:
    strip() দিয়ে leading/trailing whitespace remove করো
            ↓
    Step 4:
    Remaining inconsistencies identify করো
            ↓
    Step 5:
    Fuzzy Matching ব্যবহার করো
            ↓
    Step 6:
    Appropriate min_ratio / threshold set করো
            ↓
    Step 7:
    Output manually verify করো
            ↓
    Step 8:
    Standardized categorical data ব্যবহার করো
            ↓
    Step 9:
    প্রয়োজনে One-Hot Encoding করো


'''

pass